In [1]:
import arcpy
from arcpy.sa import *
import itertools
import pandas as pd
import os
import csv
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

In [2]:
# folder containing model coefficients
model_folder = "C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch6_CASC_Project/Dual-Risk-Repo/Carbon_Storage_Estimation/NWCA_Model"

In [3]:
# prediction depths
depths = [i for i in range(5,105,10)]

In [4]:
# load raster statistics
raster_stats = pd.read_csv(os.path.normpath(os.path.join(model_folder,'Model_Raster_Statistics.csv')))
raster_stats

,name,sd,mean
0,MAP,114.823335,1055.186234
1,MAT,2.222494,11.257228
2,Elevation,148.975113,539.655998
3,Slope,6.456477,4.404528
4,Aspect,96.392186,147.056896
5,TWI,2.294317,7.444438
6,NDWI,0.072267,-0.213836
7,NDVI,0.092037,0.300676


In [5]:
raster_names = list(raster_stats.name)[:7]
raster_names

['MAP', 'MAT', 'Elevation', 'Slope', 'Aspect', 'TWI', 'NDWI']

In [6]:
n_r = len(raster_names)

In [7]:
raster_sds = {}
raster_means = {}
for i in range(n_r):
    raster_name = raster_stats.name[i]
    raster_sds[raster_name] = raster_stats.loc[i,"sd"]
    raster_means[raster_name] = raster_stats.loc[i,"mean"]

In [8]:
raster_paths = [
    "F:/databases/climate/prism_ppt_us_30s_2020_avg_30y/ppt_avg_30y_IL_LambertNAD1983_aligned_64bit.tif",
    "F:/databases/climate/prism_tmean_us_30s_2020_avg_30y/tmean_avg_30y_IL_LambertNAD1983_aligned_64bit.tif",
    "F:/wim_rasters/dem/IL_DEM_3m_LambertNAD1983_aligned_64bit.tif",
    "F:/wim_rasters/slope/Slope_3m_LambertNAD1983_aligned_64bit.tif",
    "F:/wim_rasters/aspect/Aspect_3m_LambertNAD1983_aligned_64bit.tif",
    "F:/wim_rasters/twi/TWI_IL_Final_LambertNAD1983_Mosaic_aligned_64bit.tif",
    "F:/wim_rasters/ndwi/NDWI_IL_LambertNAD1983_mosaic_aligned_64bit.tif" #,
    #"F:/wim_rasters/ndvi/NDVI_IL_Final_LambertNAD1983_aligned_64bit.tif"
]
raster_paths

['F:/databases/climate/prism_ppt_us_30s_2020_avg_30y/ppt_avg_30y_IL_LambertNAD1983_aligned_64bit.tif',
 'F:/databases/climate/prism_tmean_us_30s_2020_avg_30y/tmean_avg_30y_IL_LambertNAD1983_aligned_64bit.tif',
 'F:/wim_rasters/dem/IL_DEM_3m_LambertNAD1983_aligned_64bit.tif',
 'F:/wim_rasters/slope/Slope_3m_LambertNAD1983_aligned_64bit.tif',
 'F:/wim_rasters/aspect/Aspect_3m_LambertNAD1983_aligned_64bit.tif',
 'F:/wim_rasters/twi/TWI_IL_Final_LambertNAD1983_Mosaic_aligned_64bit.tif',
 'F:/wim_rasters/ndwi/NDWI_IL_LambertNAD1983_mosaic_aligned_64bit.tif']

In [9]:
# load best model
model_path = os.path.normpath(os.path.join(model_folder,'Best_TC_Concentration_Model.csv'))
model_coeffs = pd.read_csv(model_path)
n_c = len(model_coeffs)
model_coeffs

,labels,values
0,(Intercept),1.250418
1,DEPTH,-1.018486
2,MAP,-0.160439
3,MAT,-0.158983
4,Elevation,0.364021
5,Slope,0.272790
6,Aspect,0.050235
7,TWI,0.120150
8,NDWI,-0.156259
9,DEPTH:MAP,0.912952


In [10]:
# get intercept and scalar depth effect
intercept = model_coeffs.loc[0,"values"]
b_depth = model_coeffs.loc[1,"values"]

In [11]:
# get main effects for single rasters
raster_coefficients = {}
for i in range(2,9):
    key = model_coeffs.loc[i,"labels"]
    raster_coefficients[key] = model_coeffs.loc[i,"values"]
raster_coefficients
n_main_effects = len(raster_coefficients)

In [12]:
# get effects for depth x raster interactions
depth_raster_coefficients = {}
depth_rasters = []
for i in range(9,13):
    key = model_coeffs.loc[i,"labels"].split(':')[1]
    depth_raster_coefficients[key] = model_coeffs.loc[i,"values"]
    depth_rasters.append(key)
n_depth_effects = len(depth_rasters)
depth_rasters

['MAP', 'MAT', 'Slope', 'NDWI']

In [13]:
# list pairs and their interaction scalar
#interaction_coefficients = {}
#for i in range(10,n_c):
#    keys = model_coeffs.loc[i,"labels"].split(':')
#    keys = tuple(keys)
#    interaction_coefficients[keys] = model_coeffs.loc[i,"values"]
#n_interaction_effects = len(interaction_coefficients)
#interaction_coefficients

In [14]:
print("Loading rasters...")
raster_objects = {raster_names[i]: Raster(raster_paths[i]) for i in range(n_r)}

Loading rasters...


In [15]:
# start with intercept
result = intercept

In [16]:
# add main effects: b_i * R_i
print("Adding main effects...")
for i in range(n_main_effects):
    name = raster_names[i]
    coef = raster_coefficients[name]
    raster = raster_objects[name]
    sd = raster_sds[name]
    mean = raster_means[name]
    term = coef * (raster - mean) / sd
    result = result + term
    print(f"  + {coef} * {name}")

Adding main effects...
  + -0.16043938248411 * MAP
  + -0.158983345280926 * MAT
  + 0.364020515274368 * Elevation
  + 0.272789947733206 * Slope
  + 0.0502345325049761 * Aspect
  + 0.120149677774087 * TWI
  + -0.156258905978323 * NDWI


In [17]:
# add interaction terms: b_ij * (R_i * R_j)
#print("Adding interaction terms...")
#for (name_i, name_j), coef in interaction_coefficients.items():
#    raster_i = raster_objects[name_i]   
#    sd_i = raster_sds[name_i]
#    mean_i = raster_means[name_i]
#    raster_j = raster_objects[name_j]
#    sd_j = raster_sds[name_j]
#    mean_j = raster_means[name_j]
#    term = coef * ((raster_i - mean_i) / sd_i) * ((raster_j - mean_j) / sd_j)
#    result = result + term
#    print(f" + {coef} * ({name_i} x {name_j})")

In [18]:
# add depth effects and write to file
for depth in depths:
    print("Adding depth effects")
    result_depth = result
    for i in range(n_depth_effects):
        name = depth_rasters[i]
        coef = depth_raster_coefficients[name]
        print(f"{coef}: {name}")
        raster = raster_objects[name]
        sd = raster_sds[name]
        mean = raster_means[name]
        term = coef * (depth/100) * (raster - mean) / sd
        result_depth = result_depth + term

    # scalar depth effect
    result_depth = Exp(result_depth + b_depth*depth/100)

    # filter anything over 100
    final_raster = Con(result_depth > 20, 20, result_depth)
    
    # save output
    output_raster = f"F:/tc_model_results/tc_concentration_{depth}cm.tif"
    print(f"\nSaving output to: {output_raster}")
    final_raster.save(output_raster)
    print(f"Finished {depth} cm")

Adding depth effects
0.912951989314251: MAP
-0.929623493207862: MAT
-0.172957724901563: Slope
0.379336669529837: NDWI

Saving output to: F:/tc_model_results/tc_concentration_5cm.tif
Finished 5 cm
Adding depth effects
0.912951989314251: MAP
-0.929623493207862: MAT
-0.172957724901563: Slope
0.379336669529837: NDWI

Saving output to: F:/tc_model_results/tc_concentration_15cm.tif
Finished 15 cm
Adding depth effects
0.912951989314251: MAP
-0.929623493207862: MAT
-0.172957724901563: Slope
0.379336669529837: NDWI

Saving output to: F:/tc_model_results/tc_concentration_25cm.tif
Finished 25 cm
Adding depth effects
0.912951989314251: MAP
-0.929623493207862: MAT
-0.172957724901563: Slope
0.379336669529837: NDWI

Saving output to: F:/tc_model_results/tc_concentration_35cm.tif
Finished 35 cm
Adding depth effects
0.912951989314251: MAP
-0.929623493207862: MAT
-0.172957724901563: Slope
0.379336669529837: NDWI

Saving output to: F:/tc_model_results/tc_concentration_45cm.tif
Finished 45 cm
Adding depth

# Estimate total C stocks

In [31]:
tc_stock_100cm = final_raster*0

In [30]:
for depth in depths:
    tc_stock_100cm = tc_stock_100cm + Raster(f"F:/tc_model_results/tc_concentration_{depth}cm.tif")*10